# Evaluating pix2pixHD on BBBC039

A conditional GAN has **no reconstruction target** during training, but at test time
we *do* have ground truth. We compare **generated** fluorescence against **real** fluorescence
using **SSIM** (Structural Similarity Index):

* **SSIM** compares **luminance**, **contrast**, and **structure** in local windows.
  Range: **[-1, 1]** (practically [0, 1] for natural images). Higher = more similar.


In [ ]:
import os, glob
os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # suppress old-GPU warnings
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
from skimage.metrics import structural_similarity as ssim


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## inference

We trained with milestone checkpoints at epochs 5, 10, 20, 40.
Run `test.py` for each checkpoint so we can compare how generation quality evolves.
*(This cell takes a few minutes — it loads four generators and runs them on the test set.)*

In [ ]:
import subprocess, sys, os, glob

checkpoints_dir = "pix2pixHD/checkpoints"
results_dir     = "pix2pixHD/results"
dataroot        = "datasets/bbbc039"
name            = "bbbc039_512"
epochs          = [5, 10, 20, 40]

# test.py writes to `<results_dir>/<name>/test_<which_epoch>/images/*.jpg` directly.
# Use sys.executable so the subprocess uses the same Python (and installed deps) as this kernel.
for ep in epochs:
    out_folder = f"{results_dir}/{name}/test_{ep}"
    if os.path.exists(out_folder) and len(glob.glob(f"{out_folder}/images/*.jpg")) > 0:
        print(f"Epoch {ep} already inferred.")
        continue
    cmd = [
        sys.executable, "pix2pixHD/test.py",
        "--name", name,
        "--dataroot", dataroot,
        "--label_nc", "0",
        "--no_instance",
        "--loadSize", "512",
        "--fineSize", "512",
        "--which_epoch", str(ep),
        "--how_many", "40",
        "--checkpoints_dir", checkpoints_dir,
        "--results_dir", results_dir,
    ]
    print(f"Running inference for epoch {ep} ...")
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:])
        print(r.stderr[-2000:])
        raise RuntimeError(f"test.py failed for epoch {ep} (exit {r.returncode})")
print("All epochs ready.")

In [ ]:
# Dynamically pick the latest available test epoch folder
name = "bbbc039_512"
results_root = f"pix2pixHD/results/{name}"
test_folders = glob.glob(os.path.join(results_root, "test_*"))
if not test_folders:
    raise FileNotFoundError("No test results found. Run the Multi-epoch inference cell first.")
# Keep only numeric-epoch folders (drop test_latest, test_random, etc.)
epoch_folders = [
    f for f in test_folders
    if os.path.basename(f).replace("test_", "").isdigit()
]
if not epoch_folders:
    raise FileNotFoundError("No numeric-epoch test folders found (only test_latest etc.).")
epoch_folders.sort(key=lambda p: int(os.path.basename(p).replace("test_", "")))
results_dir = os.path.join(epoch_folders[-1], "images")
real_dir = "datasets/bbbc039/test_B"
latest_ep = os.path.basename(epoch_folders[-1]).replace("test_", "")
print(f"Loading results from: {results_dir} (epoch {latest_ep})")

# Collect basenames
files = sorted(glob.glob(os.path.join(results_dir, "*synthesized_image.jpg")))
samples = []
for synth_path in files:
    base = os.path.basename(synth_path).replace("_synthesized_image.jpg", "")
    real_path = os.path.join(real_dir, base + ".png")
    if not os.path.exists(real_path):
        continue
    samples.append({
        "base": base,
        "mask": os.path.join(results_dir, base + "_input_label.jpg"),
        "gen":  os.path.join(results_dir, base + "_synthesized_image.jpg"),
        "real": real_path,
    })
print(f"Loaded {len(samples)} test samples from epoch {latest_ep}.")

## visual inspection

Side-by-side: mask → generated → real. Does the generator respect the mask shape? Is the texture plausible?

In [ ]:
def show_triplet(sample, ax=None):
    mask = np.array(Image.open(sample['mask']))
    gen  = np.array(Image.open(sample['gen']))
    real = np.array(Image.open(sample['real']))
    if ax is None:
        fig, ax = plt.subplots(1, 3, figsize=(12, 4))
    ax[0].imshow(mask); ax[0].set_title('Mask (input)')
    ax[1].imshow(gen);  ax[1].set_title('Generated')
    ax[2].imshow(real); ax[2].set_title('Real')
    for a in ax: a.axis('off')
    return ax

fig, axes = plt.subplots(3, 3, figsize=(12, 10))
for row, s in enumerate(samples[:3]):
    show_triplet(s, ax=axes[row])
plt.suptitle(f'Mask → Generated → Real (epoch {latest_ep}, first 3 samples)', fontsize=13)
plt.tight_layout(); plt.show()


<!-- pixel metrics removed -->


## Structural Similarity (SSIM)

SSIM compares **luminance**, **contrast**, and **structure** in local windows.
Range: **[-1, 1]** (practically [0, 1] for natural images).

In [ ]:
ssim_vals = []
for s in samples:
    gen  = np.array(Image.open(s['gen']))
    real = np.array(Image.open(s['real']))
    # SSIM on luminance (grayscale)
    g_gray = gen.mean(axis=2)
    r_gray = real.mean(axis=2)
    val = ssim(g_gray, r_gray, data_range=255.0)
    ssim_vals.append(val)

print(f'SSIM (epoch {latest_ep}) = {np.mean(ssim_vals):.4f} ± {np.std(ssim_vals):.4f}')
print(f'  min={np.min(ssim_vals):.4f}, max={np.max(ssim_vals):.4f}')


<!-- LPIPS removed -->


## Analysis — best, worst, and correlations

Which samples are easy? Which are hard? Do the metrics agree?

In [ ]:
# Attach SSIM to samples
for i, s in enumerate(samples):
    s['ssim'] = ssim_vals[i]

def show_extreme(metric, best='min'):
    sorted_samples = sorted(samples, key=lambda x: x[metric], reverse=(best=='max'))
    pick = sorted_samples[0]
    fig, ax = plt.subplots(1, 3, figsize=(12, 4))
    show_triplet(pick, ax=ax)
    title = f"{metric} = {pick[metric]:.4f} ({best}) — {pick['base']}"
    plt.suptitle(title, fontsize=12)
    plt.tight_layout(); plt.show()

print(f'=== Best / Worst by SSIM (epoch {latest_ep}) ===')
show_extreme('ssim', 'max')
show_extreme('ssim', 'min')


<!-- correlation plot removed (SSIM is sole metric) -->


### Distribution of SSIM scores

Histogram reveals whether the model is consistently good or has a long tail of failures.


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 4))

ax.hist([s['ssim'] for s in samples], bins=15, color='C2', edgecolor='k')
ax.axvline(np.mean([s['ssim'] for s in samples]), color='r', ls='--', label='mean')
ax.set_title(f'SSIM distribution (epoch {latest_ep})')
ax.set_xlabel('SSIM')
ax.legend(); ax.set_ylabel('count')
plt.tight_layout(); plt.show()

## Comparison

Compare outputs from checkpoints saved at epochs 5, 10, 20, 40.
We pick the same test sample and show how the generator improves.

In [ ]:
epochs = [5, 10, 20, 40]
name = "bbbc039_512"

# Pick the first test sample that exists across all epochs
sample_base = None
for ep in epochs:
    files = sorted(glob.glob(f"pix2pixHD/results/{name}/test_{ep}/images/*_synthesized_image.jpg"))
    if files:
        sample_base = os.path.basename(files[0]).replace('_synthesized_image.jpg', '')
        break

if sample_base is None:
    print("No inference results found. Run the Multi-epoch inference cell first.")
else:
    fig, axes = plt.subplots(len(epochs), 3, figsize=(10, 3.5*len(epochs)))
    for row, ep in enumerate(epochs):
        res_dir = f"pix2pixHD/results/{name}/test_{ep}/images"
        mask_path = f"{res_dir}/{sample_base}_input_label.jpg"
        gen_path  = f"{res_dir}/{sample_base}_synthesized_image.jpg"
        real_path = f"datasets/bbbc039/test_B/{sample_base}.png"
        mask = np.array(Image.open(mask_path)) if os.path.exists(mask_path) else np.zeros((512,512,3))
        gen  = np.array(Image.open(gen_path))  if os.path.exists(gen_path)  else np.zeros((512,512,3))
        real = np.array(Image.open(real_path)) if os.path.exists(real_path) else np.zeros((512,512,3))
        # SSIM between generated and real (grayscale luminance)
        if os.path.exists(gen_path) and os.path.exists(real_path):
            ssim_val = ssim(gen.mean(axis=2), real.mean(axis=2), data_range=255.0)
            gen_title = f"Epoch {ep}  (SSIM={ssim_val:.3f})"
        else:
            gen_title = f"Epoch {ep}  (SSIM=N/A)"
        axes[row, 0].imshow(mask); axes[row, 0].set_title("Mask")
        axes[row, 1].imshow(gen);  axes[row, 1].set_title(gen_title)
        axes[row, 2].imshow(real); axes[row, 2].set_title("Real")
        for a in axes[row]: a.axis('off')
    plt.suptitle(f"Sample: {sample_base}", fontsize=13)
    plt.tight_layout(); plt.show()

### SSIM Metric across epochs

Compute SSIM for **all** test samples at each epoch and plot the trend.

In [ ]:
from skimage.metrics import structural_similarity as ssim

epoch_stats = {ep: {'ssim': []} for ep in epochs}

for ep in epochs:
    res_dir = f'pix2pixHD/results/{name}/test_{ep}/images'
    gen_files = sorted(glob.glob(f'{res_dir}/*_synthesized_image.jpg'))
    for gen_path in gen_files:
        base = os.path.basename(gen_path).replace('_synthesized_image.jpg', '')
        real_path = os.path.join('datasets/bbbc039/test_B', base + '.png')
        if not os.path.exists(real_path):
            continue
        gen  = np.array(Image.open(gen_path))
        real = np.array(Image.open(real_path))
        # SSIM on grayscale
        g_gray = gen.mean(axis=2)
        r_gray = real.mean(axis=2)
        epoch_stats[ep]['ssim'].append(ssim(g_gray, r_gray, data_range=255.0))

fig, ax = plt.subplots(1, 1, figsize=(7, 4))
for ep in epochs:
    ax.scatter([ep]*len(epoch_stats[ep]['ssim']), epoch_stats[ep]['ssim'], alpha=0.4)
ax.plot(epochs, [np.mean(epoch_stats[ep]['ssim']) for ep in epochs], 'k-o', label='mean')
ax.set_title('SSIM across epochs (higher → better)'); ax.set_xlabel('Epoch')
ax.set_ylabel('SSIM')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Random mask generation

How does the generator respond to **synthetic masks** it never saw during training?
We create random binary masks (ellipses and circles) and run them through the best epoch.

In [ ]:
import numpy as np
from PIL import Image, ImageDraw
import random, os, subprocess, sys

random.seed(42)
np.random.seed(42)

def make_random_mask(size=512, n_objs=8):
    img = Image.new('L', (size, size), 0)
    draw = ImageDraw.Draw(img)
    for _ in range(n_objs):
        x = random.randint(20, size-20)
        y = random.randint(20, size-20)
        rx = random.randint(10, 40)
        ry = random.randint(10, 40)
        draw.ellipse([x-rx, y-ry, x+rx, y+ry], fill=255)
    return img

rand_dir = 'random_masks'
os.makedirs(f'{rand_dir}/test_A', exist_ok=True)
os.makedirs(f'{rand_dir}/test_B', exist_ok=True)

for i in range(5):
    mask = make_random_mask()
    mask.save(f'{rand_dir}/test_A/rand_{i:02d}.png')
    Image.new('RGB', (512, 512), (0,0,0)).save(f'{rand_dir}/test_B/rand_{i:02d}.png')

# Pick the best available epoch (exclude 'latest', sort numerically)
ckpt_files = [f for f in glob.glob('pix2pixHD/checkpoints/bbbc039_512/*_net_G.pth') if 'latest' not in f]
best_ep = 'latest'
if ckpt_files:
    best_ep = sorted([int(os.path.basename(f).replace('_net_G.pth', '')) for f in ckpt_files])[-1]
    best_ep = str(best_ep)

# pix2pixHD saves to test_<epoch> when --which_epoch is a number
out_dir = f'pix2pixHD/results/bbbc039_512/test_{best_ep}/images'

# Use sys.executable so the subprocess uses the same Python (and installed deps) as this kernel.
cmd = [
    sys.executable, 'pix2pixHD/test.py',
    '--name', 'bbbc039_512',
    '--dataroot', rand_dir,
    '--label_nc', '0',
    '--no_instance',
    '--loadSize', '512',
    '--fineSize', '512',
    '--which_epoch', best_ep,
    '--how_many', '5',
    '--checkpoints_dir', 'pix2pixHD/checkpoints',
    '--results_dir', 'pix2pixHD/results',
]
r = subprocess.run(cmd, capture_output=True, text=True)
if r.returncode != 0:
    print(r.stdout[-2000:])
    print(r.stderr[-2000:])
    raise RuntimeError(f"test.py failed on random masks (exit {r.returncode})")

# Visualise
fig, axes = plt.subplots(2, 5, figsize=(14, 5))
for i in range(5):
    m = np.array(Image.open(f'{rand_dir}/test_A/rand_{i:02d}.png'))
    g = np.array(Image.open(f'{out_dir}/rand_{i:02d}_synthesized_image.jpg'))
    axes[0, i].imshow(m, cmap='gray'); axes[0, i].set_title('Random mask')
    axes[1, i].imshow(g);          axes[1, i].set_title('Generated')
    for a in axes[:, i]: a.axis('off')
plt.suptitle(f'Random masks → Generated fluorescence (epoch {best_ep})', fontsize=13)
plt.tight_layout(); plt.show()

## Discussion

1. **Deterministic output:** pix2pixHD injects noise during generation. If you re-run `test.py` on the same mask,
   do you get the same image? What does that imply?

## Takeaway

Evaluating generative models requires **more effort**:
* We need a combination of metrics and manual inspection

**look at the generated samples**, then quantify.